# G2 D6-TV2 — Evaluation E1/E2
**Bản notebook này: NOT RUN trên GPU Colab.** Chọn GPU runtime (T4 hoặc GPU Colab Pro).
Chỉ nghiệm thu khi đủ **8/8 E1 + 8/8 E2**, cùng protocol và 8 cặp hợp lệ.
Notebook không train. Mọi inference/metric/coverage đi qua `scripts/eval_g2.py`.

Setup seed **42** chỉ dùng cho phiên notebook. TRAIN/DEV seed, loss, threshold và budget
đọc từ checkpoint đã huấn luyện; không đổi sang 42 khi đánh giá.
Synthetic DEV do loader hiện có tạo từ VALIDATION/good với protocol/seed đã khóa,
không tạo dữ liệu thay thế và không đọc TEST.

Commit/push phần evaluator lên `g2/member2` trước khi chạy Colab. Có thể nhập SHA đầy đủ ở
`MSILA_COMMIT`; để trống sẽ chốt HEAD của branch đúng một lần rồi checkout detached SHA đó.
Sau phiên đầu, dùng SHA đã in/lưu để lặp lại. DINO checkout phải đúng revision đã dùng khi train.

## 1. Setup — GPU, repo, dependencies, Drive
Chạy từ đầu bằng **Run all** sau khi sửa các đường dẫn ở phần 2. Không bắt buộc riêng T4.
Venv giữ PyTorch CUDA của Colab nếu đã đáp ứng requirements.

In [ ]:
# @title Cell 1 — Setup
from pathlib import Path
import json, os, shutil, subprocess, sys
from IPython.display import HTML, Image as DisplayImage, display
from google.colab import drive

MSILA_COMMIT = '' # @param {type:'string'}
DINO_COMMIT = '6876159a11b4df116f30f667f8c9888617df0751' # @param {type:'string'}
SETUP_SEED = 42
WORK = Path('/content/msila_g2_evaluation')
REPO, DINO = WORK/'msila', WORK/'dinov3'
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)
drive.mount('/content/drive')

def checkout(url, destination, branch, revision):
    if not (destination/'.git').is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, url, str(destination)], check=True)
    if subprocess.check_output(['git', '-C', str(destination), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError(f'BLOCKED: checkout có thay đổi: {destination}')
    if subprocess.check_output(['git', '-C', str(destination), 'remote', 'get-url', 'origin'], text=True).strip() != url:
        raise RuntimeError(f'BLOCKED: repo origin không đúng: {destination}')
    if not revision:
        revision = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    if len(revision) != 40 or any(c not in '0123456789abcdef' for c in revision):
        raise ValueError('BLOCKED: commit phải là SHA 40 ký tự')
    subprocess.run(['git', '-C', str(destination), 'fetch', '--depth', '1', 'origin', revision], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', revision], check=True)
    actual = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != revision:
        raise RuntimeError('BLOCKED: checkout SHA mismatch')
    return actual

MSILA_COMMIT = checkout('https://github.com/haduckien-raccoon/msila.git', REPO, 'g2/member2', MSILA_COMMIT)
DINO_COMMIT = checkout('https://github.com/facebookresearch/dinov3.git', DINO, 'main', DINO_COMMIT)
if not (REPO/'scripts/eval_g2.py').is_file():
    raise FileNotFoundError('BLOCKED: commit chưa chứa eval_g2.py; commit/push code D6-TV2 rồi chạy lại')
VENV = WORK/'venv'
if not (VENV/'bin/python').is_file():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', str(REPO/'requirements.txt')], check=True)
ENV = dict(os.environ, PYTHONHASHSEED=str(SETUP_SEED), PYTHONUNBUFFERED='1', MPLBACKEND='Agg',
           MPLCONFIGDIR=str(WORK/'matplotlib'), CUBLAS_WORKSPACE_CONFIG=':4096:8')
ENVIRONMENT = json.loads(subprocess.check_output([PYTHON, '-c',
    "import json,platform,torch; print(json.dumps(dict(python=platform.python_version(),torch=str(torch.__version__),cuda=torch.version.cuda,cuda_available=torch.cuda.is_available(),gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)))"],
    cwd=REPO, env=ENV, text=True))
if not ENVIRONMENT['cuda_available']:
    raise RuntimeError('BLOCKED: CUDA không khả dụng; chọn GPU runtime')
print(json.dumps(dict(environment=ENVIRONMENT, msila_commit=MSILA_COMMIT, dino_commit=DINO_COMMIT,
                     setup_seed=SETUP_SEED, gpu_evaluation='NOT RUN'), indent=2))
subprocess.run([PYTHON, 'scripts/eval_g2.py', '--help'], cwd=REPO, env=ENV, check=True)
sys.path.insert(0, str(REPO))
from scripts.eval_g2 import CATEGORIES, prepare_archives, copy_atomic, sync_tree, run_logged

## 2. Data / Checkpoint — preflight đủ tám category
Sửa `DRIVE_PROJECT`, tên 8 archive, `E1_ON_DRIVE`, `D6_STUDY_ON_DRIVE`, pretrained weights.
**Copy đủ 8 `.tar.gz` từ Drive sang Colab trước rồi mới giải nén local.** Chỉ giải nén TRAIN/good
và VALIDATION/good; không giải nén TEST. Source order/hash phải trùng checkpoint.

E1: `<category>/{best.pt,last.pt,metrics.json}` và hai SHA sidecar, để xác minh budget cuối.
E2: giữ nguyên toàn bộ thư mục `full/`
TV1 xuất ra: `E2/<category>/{best.pt,last.pt,metrics.json}` và các SHA sidecar,
`adapter_selection_lock.json`, `protocol_lock.json`, `adapter_screen/adapter_r*_d*/<category>/`
(đủ 72 run và checkpoint/evidence). Lock JSON đơn lẻ không đủ xác minh.
Checkpoint/pretrained weights phải là artifact thật; thiếu sẽ dừng với BLOCKED.
E1/E2 khác backbone/decoder/seed/budget/sources sẽ bị chặn so sánh.

In [ ]:
# @title Cell 2 — Data / Checkpoint
DRIVE_PROJECT = '/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9' # @param {type:'string'}
BACKBONE = 'dinov3_vitb16' # @param {type:'string'}
PRETRAINED_ON_DRIVE = '' # @param {type:'string'}
E1_ON_DRIVE = '' # @param {type:'string'}
D6_STUDY_ON_DRIVE = '' # @param {type:'string'}
RUN_TAG = 'D6_E1_E2' # @param {type:'string'}
PROJECT = Path(DRIVE_PROJECT)
ARCHIVE_DIR = PROJECT/'data'
ARCHIVE_NAMES = {category: f'{category}.tar.gz' for category in CATEGORIES} # Sửa tên archive nếu khác.
E1_SOURCE = Path(E1_ON_DRIVE) if E1_ON_DRIVE.strip() else PROJECT/'outputs/G1/E1'
D6_SOURCE = Path(D6_STUDY_ON_DRIVE) if D6_STUDY_ON_DRIVE.strip() else PROJECT/'outputs/G2/D6'/BACKBONE
PRETRAINED_NAMES = {'dinov3_vits16':'dinov3_vits16_pretrain_lvd1689m-08c60483.pth',
                   'dinov3_vits16plus':'dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth',
                   'dinov3_vitb16':'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth',
                   'dinov3_vitl16':'dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth',
                   'dinov3_vith16plus':'dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth'}
WEIGHT_SOURCE = Path(PRETRAINED_ON_DRIVE) if PRETRAINED_ON_DRIVE.strip() else PROJECT/'weights'/PRETRAINED_NAMES[BACKBONE]
OUTPUT = WORK/'outputs'
DRIVE_OUTPUT = PROJECT/'outputs/G2/evaluation'/RUN_TAG/MSILA_COMMIT[:12]
OUTPUT.mkdir(parents=True, exist_ok=True)
sync_tree(DRIVE_OUTPUT, OUTPUT) # Khôi phục category đã hoàn tất sau khi Colab ngắt phiên.
SETUP_MANIFEST = dict(msila_commit=MSILA_COMMIT, dino_commit=DINO_COMMIT, setup_seed=SETUP_SEED,
                      environment=ENVIRONMENT, original_e1_root=str(E1_SOURCE), original_d6_root=str(D6_SOURCE))
(OUTPUT/'notebook_environment.json').write_text(json.dumps(SETUP_MANIFEST, indent=2))
missing = [str(p) for p in (WEIGHT_SOURCE, E1_SOURCE, D6_SOURCE/'full') if not p.exists()]
missing += [str(E1_SOURCE/c/'best.pt') for c in CATEGORIES if not (E1_SOURCE/c/'best.pt').is_file()]
missing += [str(D6_SOURCE/'full/E2'/c/'best.pt') for c in CATEGORIES if not (D6_SOURCE/'full/E2'/c/'best.pt').is_file()]
if missing:
    (OUTPUT/'notebook_status.json').write_text(json.dumps(dict(status='BLOCKED', missing=missing), indent=2))
    sync_tree(OUTPUT, DRIVE_OUTPUT)
    raise FileNotFoundError('BLOCKED: thiếu input thật:\n'+'\n'.join(missing))
DATA = prepare_archives(ARCHIVE_DIR, ARCHIVE_NAMES, WORK/'archives', WORK/'dataset')
WEIGHTS = copy_atomic(WEIGHT_SOURCE, WORK/'weights'/WEIGHT_SOURCE.name)
E1_ROOT, FULL = WORK/'checkpoints/E1', WORK/'checkpoints/D6/full'
sync_tree(E1_SOURCE, E1_ROOT)
sync_tree(D6_SOURCE/'full', FULL)
COMMON_ARGS = ['--experiments','E1','E2','--e1-root',str(E1_ROOT),'--e2-root',str(FULL/'E2'),
               '--selection-lock',str(FULL/'adapter_selection_lock.json'),'--data-root',str(DATA),
               '--repo-dir',str(DINO),'--weights',str(WEIGHTS),'--device','cuda','--output-root',str(OUTPUT)]

def run_eval(arguments, stage):
    return run_logged([PYTHON,'scripts/eval_g2.py',*arguments], cwd=REPO, output_root=OUTPUT,
                      drive_output=DRIVE_OUTPUT, stage=stage, env=ENV)

PREFLIGHT_CODE = run_eval([*COMMON_ARGS,'--categories','all','--preflight'], 'preflight')
if PREFLIGHT_CODE:
    raise RuntimeError(f'BLOCKED/FAIL: preflight exit {PREFLIGHT_CODE}; xem Drive preflight/coverage.json và preflight.log')
print('Preflight READY; chưa có metric nghiệm thu. Seed DEV/training giữ nguyên trong checkpoint.')

## 3. Smoke — một category, ba mẫu DEV gốc
Kiểm tra restore heads, extraction, native map, tiny/mixed AU-PRO, Dice và P99.
Output riêng `smoke/`; **SMOKE_PASS không được tính vào coverage nghiệm thu**.

In [ ]:
# @title Cell 3 — Smoke test
SMOKE_CATEGORY = 'rice' # @param {type:'string'}
SMOKE_CODE = run_eval([*COMMON_ARGS,'--categories',SMOKE_CATEGORY,'--smoke'], 'smoke')
if SMOKE_CODE:
    raise RuntimeError(f'BLOCKED/FAIL: smoke exit {SMOKE_CODE}; xem smoke/summary.json và smoke.log trên Drive')
print('SMOKE_PASS; smoke không đóng góp vào coverage nghiệm thu.')

## 4. Full evaluation — E1/E2 đủ tám category
`--resume` chỉ skip kết quả còn hợp lệ: checkpoint/config/source/code hash, metric,
QA và toàn bộ native map/mask đã lưu. Run thiếu/khác inputs sẽ không được tính PASS.
Muốn chạy riêng một category: đổi `FULL_CATEGORIES`; vẫn giữ đủ 8 hàng báo cáo,
không có macro 8 category khi coverage chưa đủ. Đồng bộ Drive mỗi 30 giây và cuối run.

In [ ]:
# @title Cell 4 — Full evaluation / resume
FULL_CATEGORIES = 'all' # @param {type:'string'}
FULL_CODE = run_eval([*COMMON_ARGS,'--categories',FULL_CATEGORIES,'--resume'], 'full')
print('Evaluator exit:', FULL_CODE, '(0=PASS; 1=FAIL; 2=BLOCKED/incomplete). Xem kết quả thực ở phần 5.')

## 5. Results — bảng, biểu đồ, native maps và export
Không tính lại metric trong notebook. Bảng CSV luôn giữ đủ 8 category với ô trống cho run thiếu.
Macro chỉ xuất khi đủ 8 cặp có cùng protocol và cùng exact synthetic DEV.
Toàn bộ CSV/JSON/native `.npy`/examples/config/Git/GPU/log được đồng bộ Google Drive.

In [ ]:
# @title Cell 5 — Results / Export
import csv, html
REPORT_CODE = run_eval([*COMMON_ARGS,'--categories','all','--report-only'], 'report')
summary = json.loads((OUTPUT/'summary.json').read_text())
print(json.dumps({k:v for k,v in summary.items() if k != 'records'}, indent=2))
with (OUTPUT/'metrics_E1_E2.csv').open(newline='') as handle:
    rows = list(csv.DictReader(handle))
columns = ['category','E1_status','E2_status','E1_synthetic_dev_aupro_0_05',
           'E2_synthetic_dev_aupro_0_05','E2_minus_E1_aupro_0_05','pair_status']
table = '<table><tr>'+''.join('<th>'+html.escape(c)+'</th>' for c in columns)+'</tr>'
table += ''.join('<tr>'+''.join('<td>'+html.escape(row.get(c,''))+'</td>' for c in columns)+'</tr>' for row in rows)+'</table>'
display(HTML(table))
display(DisplayImage(filename=str(OUTPUT/'E2_minus_E1_aupro.png')))
for experiment in ('E1','E2'):
    examples = sorted((OUTPUT/experiment).glob('*/examples/*/comparison.png'))
    for example in examples[:2]:
        print(experiment, example.parent.name, 'Original / Synthetic input / Exact GT / Probability / Overlay')
        display(DisplayImage(filename=str(example)))
if summary['status'] == 'PASS' and summary['complete'] and summary['paired_pass'] == 8:
    print('PASS: E1 8/8, E2 8/8, paired 8/8. Macro:', summary['macro_mean_8categories'])
else:
    print(summary['status'], ': thiếu kết quả hợp lệ; KHÔNG tuyên bố nghiệm thu đủ 8 category.')
sync_tree(OUTPUT, DRIVE_OUTPUT)
print('Đã lưu:', DRIVE_OUTPUT)